In [2]:
import torch
import torch.nn.functional as F


In [42]:
margin = torch.load('stats/0/margin_224_288.pt')
conf = torch.load('stats/0/conf_32_96.pt')
unmask = torch.load('stats/0/unmask_32_96.pt')
attn = torch.load('stats/1/attn_96_160.pt')

In [46]:
attn.shape
# conf[:,:,slice(1,3)].shape

torch.Size([64, 32, 1, 64])

In [3]:
L=12
torch.full((L,), -1, dtype=torch.long)

tensor([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])

In [ ]:
a = torch.tensor([1,2,3])
print(type(a))
print(type(a) is torch.Tensor)
a.numpy().tolist()
torch.arange(5)[0:3]
a = [torch.arange(5)[0:3], torch.arange(5)[0:3]]
torch.stack(a, dim=0)

tensor([0, 1, 2])

In [ ]:
def get_margin_p(logits, mask_mask, idx_a=0, idx_b=1):   # returns p[rank a] - p[rank b], rank 0 = top-1
    p = F.softmax(logits.to(torch.float64), dim=-1)   # [N, V]
    p_sorted, _ = torch.sort(p, dim=-1, descending=True)    # rank 0 = largest prob
    margin_p = p_sorted[:, idx_a] - p_sorted[:, idx_b]          # [N]
    neg_inf = torch.tensor(torch.finfo(p.dtype).min, device=p.device, dtype=p.dtype)
    margin_p = torch.where(mask_mask.squeeze(-1), margin_p, neg_inf)
    return margin_p
# end

logits = torch.tensor([[7,8,9],[1,2,3],[4,5,6]])
mask_mask = torch.tensor([[1],[0],[1]], dtype=bool)
print(get_margin_p(logits, mask_mask, 0, 1))
